# Introduction to Lakehouse Runtime Catalog for Iceberg with credential vending

This tutorial covers the credential vending authentication mode. Its very basic just to demonstrate the concept.

# 1. Setup

Configure environment variables. Provide your project ID and a [region](https://cloud.google.com/bigquery/docs/locations#regions) to store your resources, such as `us-central1`. Also create a serverless interactive Spark session.

## 1.1. Define variables

In [ ]:
PROJECT_ID_LIST=!gcloud config list --format "value(core.project)" 2>/dev/null
PROJECT_ID=PROJECT_ID_LIST[0]
PROJECT_NBR= ! gcloud projects describe $PROJECT_ID | grep projectNumber | cut -d':' -f2 | xargs
PROJECT_NBR=PROJECT_NBR[0]
LOCATION = "us-central1"
STAGE_BUCKET_NAME = f"froyo-lakehouse-staging-{PROJECT_NBR}"
ICEBERG_LAKEHOUSE_BUCKET_NAME = f"froyo_iceberg_lakehouse_catalog_cv_{PROJECT_NBR}"
ICEBERG_CATALOG_NAME="froyo_iceberg_catalog_cv"
ICEBERG_NAMESPACE="froyo_ns"
APP_NAME="froyo_app"

## 1.2. Create a lakehouse bucket for use with the catalog

In [ ]:
!pip install google-cloud-storage

In [ ]:
from google.cloud import storage

def create_gcs_bucket(bucket_name, location="US", storage_class="STANDARD"):
    """
    Creates a new GCS bucket with the specified name, location, and storage class.
    """
    # Initialize the GCS client
    # It will automatically pick up your credentials from the environment
    storage_client = storage.Client()

    # Create a new bucket object (this doesn't create it in the cloud yet)
    bucket = storage_client.bucket(bucket_name)

    # Set the storage class (e.g., STANDARD, NEARLINE, COLDLINE, ARCHIVE)
    bucket.storage_class = storage_class

    try:
        # Actually create the bucket in the cloud
        new_bucket = storage_client.create_bucket(bucket, location=location)

        print(f"Success! Created bucket: {new_bucket.name}")
        print(f"Location: {new_bucket.location}")
        print(f"Storage Class: {new_bucket.storage_class}")

        return new_bucket

    except Exception as e:
        print(f"Failed to create bucket '{bucket_name}': {e}")
        return None


In [ ]:
create_gcs_bucket(ICEBERG_LAKEHOUSE_BUCKET_NAME,LOCATION,"STANDARD" )

# 2. Create a spark session with Iceberg catalog configuration

In [ ]:
from google.cloud.dataproc_spark_connect import DataprocSparkSession
from google.cloud.dataproc_v1 import Session
from pyspark.sql import functions as F

REST_API_VERSION="v1beta"

# Create the Dataproc Serverless session.
s8s_spark_session = Session()

# Serverless runtime at authoring was 3.0 with Iceberg 1.10
s8s_spark_session.runtime_config.properties["spark.sql.defaultCatalog"] = ICEBERG_CATALOG_NAME
s8s_spark_session.runtime_config.properties[f"spark.sql.catalog.{ICEBERG_CATALOG_NAME}"] = "org.apache.iceberg.spark.SparkCatalog"
s8s_spark_session.runtime_config.properties[f"spark.sql.catalog.{ICEBERG_CATALOG_NAME}.type"] = "rest"
s8s_spark_session.runtime_config.properties[f"spark.sql.catalog.{ICEBERG_CATALOG_NAME}.uri"] = f"https://biglake.googleapis.com/iceberg/{REST_API_VERSION}/restcatalog"
s8s_spark_session.runtime_config.properties[f"spark.sql.catalog.{ICEBERG_CATALOG_NAME}.warehouse"] = f"gs://{ICEBERG_LAKEHOUSE_BUCKET_NAME}"
s8s_spark_session.runtime_config.properties[f"spark.sql.catalog.{ICEBERG_CATALOG_NAME}.io-impl"] = "org.apache.iceberg.gcp.gcs.GCSFileIO"
s8s_spark_session.runtime_config.properties[f"spark.sql.catalog.{ICEBERG_CATALOG_NAME}.header.x-goog-user-project"] = PROJECT_ID
s8s_spark_session.runtime_config.properties[f"spark.sql.catalog.{ICEBERG_CATALOG_NAME}.rest.auth.type"] = "org.apache.iceberg.gcp.auth.GoogleAuthManager"
s8s_spark_session.runtime_config.properties["spark.sql.extensions"] = "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
s8s_spark_session.runtime_config.properties[f"spark.sql.catalog.{ICEBERG_CATALOG_NAME}.rest-metrics-reporting-enabled"] = "false"
s8s_spark_session.runtime_config.properties[f"spark.sql.catalog.{ICEBERG_CATALOG_NAME}.header.X-Iceberg-Access-Delegation"] = "vended-credentials"
s8s_spark_session.runtime_config.properties[f"spark.sql.catalog.{ICEBERG_CATALOG_NAME}.gcs.oauth2.refresh-credentials-endpoint"] = "https://oauth2.googleapis.com/token"
s8s_spark_session.runtime_config.properties["dataproc.lineage.enabled"] = "true"
s8s_spark_session.runtime_config.properties["spark.openlineage.transport.type"] = "gcplineage"
s8s_spark_session.runtime_config.properties["spark.extraListeners"] = "io.openlineage.spark.agent.OpenLineageSparkListener"
s8s_spark_session.runtime_config.properties["spark.sql.repl.eagerEval.enabled"] = "True" # Property values should be strings
s8s_spark_session.runtime_config.properties["spark.openlineage.namespace"] = "froyo_spark_jobs"
s8s_spark_session.runtime_config.properties["spark.log.level.io.openlineage"] = "DEBUG"

#s8s_spark_session.runtime_config.properties["spark.driver.extraJavaOptions"] = "-Dlog4j.level=DEBUG" # Example for driver
#s8s_spark_session.runtime_config.properties["spark.executor.extraJavaOptions"] = "-Dlog4j.level=DEBUG" # Example for executors
#s8s_spark_session.runtime_config.properties["spark.hadoop.log4j.rootLogger"] = "DEBUG,CONSOLE" # A more standard way for Spark 3.x+




spark = (DataprocSparkSession.builder
    .appName(APP_NAME)
    .dataprocSessionConfig(s8s_spark_session)
    .getOrCreate())



## 2. Create an Iceberg namespace

In [ ]:
spark.sql(f"SHOW CATALOGS;").show(truncate=False)

In [ ]:
spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {ICEBERG_NAMESPACE};").show(truncate=False)
spark.sql(f"USE {ICEBERG_NAMESPACE};")

In [ ]:
spark.sql("SHOW NAMESPACES;").show(truncate=False)

In [ ]:
spark.sql(f"SHOW TABLES IN {ICEBERG_NAMESPACE}").show(truncate=False)

In [ ]:
# Drop any existing tables in case of a rerun

# 1. Get the dataframe containing the tables
tables_df = spark.sql(f"SHOW TABLES IN {ICEBERG_NAMESPACE}")

# 2. Collect rows to the driver
# (SHOW TABLES outputs columns: 'namespace', 'tableName', and 'isTemporary')
tables_list = tables_df.collect()

print(f"Found {len(tables_list)} targets in 'froyo_ns'. Starting iterative drop...")

# 3. Loop and drop
for row in tables_list:
    table_name = row['tableName']
    is_temp = row['isTemporary']

    # Fully qualify the name to ensure you drop from the correct namespace
    fq_name = f"{ICEBERG_NAMESPACE}.{table_name}"

    try:
        if is_temp:
            # If it's a temporary view, use DROP VIEW
            print(f"Dropping temporary view: {table_name}")
            spark.sql(f"DROP TEMPORARY VIEW IF EXISTS {table_name}")
        else:
            # Standard or Iceberg table
            print(f"Dropping table: {fq_name}")
            spark.sql(f"DROP TABLE IF EXISTS {fq_name}")

    except Exception as e:
        print(f"⚠️ Failed to drop {table_name}: {e}")

print("Iterative drop process complete.")

# 3. Create bronze layer - plain old parquet tables


In [ ]:
## 3.1. CUSTOMER MASTER

# This uses the local Spark catalog and not the Iceber catalog in Lakehouse runtime catalog service

# Load parquet data from GCS staging bucket and persist to bronze (raw) layer with data in full fidelity
customer_stage_df = spark.read.format("parquet").option("inferschema",True).load(f"gs://{STAGE_BUCKET_NAME}/froyo-data/customers")


# Write to bronze layer / raw layer
customer_stage_df.coalesce(1).write.mode("overwrite").parquet(f"gs://{ICEBERG_LAKEHOUSE_BUCKET_NAME}/froyo-raw/bronze/customers")


# Create a temporary table
spark.read.format("parquet").option("inferschema",True).load(f"gs://{ICEBERG_LAKEHOUSE_BUCKET_NAME}/froyo-raw/bronze/customers").createOrReplaceTempView("b_customer_master")


# Run some quick stats
spark.sql(f"select distinct status, count(*) customers from b_customer_master group by status").show(truncate=False)
spark.sql(f"select count(*) customers from b_customer_master").show(truncate=False)
spark.sql(f"select count(distinct *) distinct_customers from b_customer_master").show(truncate=False)
spark.sql(f"select * from b_customer_master limit 2").show(truncate=False)


In [ ]:
## 3.2. regions

# Load parquet data from GCS staging bucket and persist to bronze (raw) layer with data in full fidelity
regions_stage_df = spark.read.format("parquet").option("header", True).option("inferschema",True).load(f"gs://{STAGE_BUCKET_NAME}/froyo-data/regions")

# Write to bronze layer / raw layer
regions_stage_df.coalesce(1).write.mode("overwrite").parquet(f"gs://{ICEBERG_LAKEHOUSE_BUCKET_NAME}/froyo-raw/bronze/regions")

# Create a temporary table
spark.read.format("parquet").option("inferschema",True).load(f"gs://{ICEBERG_LAKEHOUSE_BUCKET_NAME}/froyo-raw/bronze/regions").createOrReplaceTempView("b_regions")

# Some quick stats
spark.sql(f"select count(*) regions from b_regions").show(truncate=False)
spark.sql(f"select count(distinct *) distinct_regions from b_regions").show(truncate=False)
spark.sql(f"select * from b_regions limit 2").show(truncate=False)

# 4. Create silver layer - Iceberg tables in credentials vending mode

## 4.1. Check the namespace for tables

In [ ]:
spark.sql("show tables in froyo_iceberg_catalog_cv.froyo_ns").show(truncate=False)

## 4.2. Create customer master silver layer table



Rules we will apply to create s_customer_master
1. Deduplicate
2. Explode the json
3. Get the region details into the customer table
4. Ensure we dont have too many small files

In [ ]:
# Coalescing as we have VERY small data for the purpose of a hands on lab.
# Consider other Spark optimizations for optimal file sizing without skew.

silver_df=spark.sql("select c.customer_id, c.customer_nm,get_json_object(c.demographics, '$.age_bracket') AS age_bracket, " \
                    "get_json_object(c.demographics, '$.income') AS income,r.city, r.state_province as state_cd, " \
                    "r.zip_code as zip_cd, r.country as country_cd,c.status,c.consent_ts " \
                    "from b_customer_master c left outer join b_regions r on c.region_id=r.region_id").dropDuplicates()
silver_df.show(2, truncate=False)
silver_df.count()
silver_df.printSchema()
silver_df.coalesce(1).write.format("iceberg").mode("overwrite").saveAsTable("froyo_ns.s_customer_master")


In [ ]:
spark.sql("show tables in froyo_ns").show(truncate=False)

# 5. Create platinum layer- customer segmentation by age

We are running this look at lineage in Knowledge Catalog.

In [ ]:
import matplotlib.pyplot as plt

# Generate customer segmentation report by customer age bracket
customer_segmentation_df = spark.sql("""
    SELECT
        age_bracket as customer_age_bracket,
        COUNT(DISTINCT customer_id) AS number_of_customers
    FROM froyo_ns.s_customer_master
    GROUP BY age_bracket
    ORDER BY age_bracket
""")

# Persist to Iceberg table
customer_segmentation_df.writeTo("froyo_ns.p_rdm_customer_segmentation_by_age") \
    .tableProperty("write.format.default", "parquet") \
    .createOrReplace()

# Display the result (optional)
customer_segmentation_df.show()


In [ ]:
spark.sql("SHOW TABLES IN froyo_ns").show(truncate=False)

## This concludes the tutorial. Proceed back to the lab manual.